# 02 Train Model

Train the Seq2Seq Transformer with the configured dataset and SentencePiece tokenizers. This notebook saves `checkpoints/latest.pt` after every epoch and updates `checkpoints/best.pt` when validation loss improves.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

## Colab Setup

Run the next cell first when using Google Colab. It clones the repository, installs project dependencies from `pyproject.toml`, and moves the working directory to the repository root.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/songhahyun/seq2seq_transformer_model.git"
REPO_DIR = Path("/content/seq2seq_transformer_model")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "dev", "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", ".[notebooks]"], check=True)
else:
    print("Not running in Colab; skipping clone/install.")

print(f"cwd: {Path.cwd()}")

In [ ]:
import torch.optim as optim

from src.config import Config
from src.data_pipeline import (
    create_dataloaders,
    extract_pairs,
    load_ko_en_dataset,
    maybe_take_subset,
    prepare_tokenizers,
    set_seed,
    split_pairs,
)
from src.model_utils import build_model
from src.train import create_loss_fn, save_checkpoint, train_one_epoch, validate_one_epoch

config = Config()
set_seed(config.random_seed)

print(f"device: {config.device}")
print(f"dataset: {config.dataset_name}")
print(f"src tokenizer: {config.sp_model_path_src}")
print(f"tgt tokenizer: {config.sp_model_path_tgt}")

## Load and split dataset

In [ ]:
dataset = load_ko_en_dataset(
    config.dataset_name,
    split=config.train_split,
    hf_token=config.hf_token,
)
pairs = extract_pairs(dataset, src_col="ko", tgt_col="en")

train_pairs, valid_pairs, test_pairs = split_pairs(
    pairs,
    valid_ratio=config.valid_ratio,
    test_ratio=config.test_ratio,
    seed=config.random_seed,
)

train_pairs = maybe_take_subset(train_pairs, config.train_subset_size)
valid_pairs = maybe_take_subset(valid_pairs, config.valid_subset_size)
test_pairs = maybe_take_subset(test_pairs, config.test_subset_size)

print(f"total pairs: {len(pairs)}")
print(f"train pairs: {len(train_pairs)}")
print(f"valid pairs: {len(valid_pairs)}")
print(f"test pairs : {len(test_pairs)}")

## Prepare tokenizers and dataloaders

SentencePiece files are loaded from `data/spm_*.model` when present. If they do not exist, they are trained from `train_pairs` and saved under `data/`.

In [ ]:
sp_src, sp_tgt = prepare_tokenizers(train_pairs, config)

train_loader, valid_loader, test_loader = create_dataloaders(
    train_pairs,
    valid_pairs,
    test_pairs,
    sp_src,
    sp_tgt,
    config,
)

print(f"src vocab size: {sp_src.get_piece_size()}")
print(f"tgt vocab size: {sp_tgt.get_piece_size()}")
print(f"epochs: {config.num_epochs}")
print(f"batch size: {config.batch_size}")
print(f"learning rate: {config.lr}")
print(f"train batches : {len(train_loader)}")
print(f"valid batches : {len(valid_loader)}")
print(f"test batches : {len(test_loader)}")

## Build model

In [ ]:
model = build_model(
    config=config,
    src_vocab_size=sp_src.get_piece_size(),
    tgt_vocab_size=sp_tgt.get_piece_size(),
)

optimizer = optim.Adam(model.parameters(), lr=config.lr)
criterion = create_loss_fn(config.pad_id)

num_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {num_params:,}")

## Train and save checkpoints

In [ ]:
history = []
best_valid_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(config.num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        config.device,
    )
    valid_loss = validate_one_epoch(
        model,
        valid_loader,
        criterion,
        config.device,
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
    })

    print(
        f"[Epoch {epoch + 1}/{config.num_epochs}] "
        f"train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f}"
    )

    latest_checkpoint_path = f"{config.checkpoint_dir}/latest.pt"
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        config=config,
        epoch=epoch + 1,
        train_loss=train_loss,
        valid_loss=valid_loss,
        src_vocab_size=sp_src.get_piece_size(),
        tgt_vocab_size=sp_tgt.get_piece_size(),
        path=latest_checkpoint_path,
    )
    print(f"saved checkpoint: {latest_checkpoint_path}")

    improved = valid_loss < best_valid_loss - config.early_stopping_min_delta
    if improved:
        best_valid_loss = valid_loss
        epochs_without_improvement = 0
        best_checkpoint_path = f"{config.checkpoint_dir}/best.pt"
        save_checkpoint(
            model=model,
            optimizer=optimizer,
            config=config,
            epoch=epoch + 1,
            train_loss=train_loss,
            valid_loss=valid_loss,
            src_vocab_size=sp_src.get_piece_size(),
            tgt_vocab_size=sp_tgt.get_piece_size(),
            path=best_checkpoint_path,
        )
        print(f"saved best checkpoint: {best_checkpoint_path}")
    else:
        epochs_without_improvement += 1
        print(
            "no validation improvement "
            f"({epochs_without_improvement}/{config.early_stopping_patience})"
        )

        if epochs_without_improvement >= config.early_stopping_patience:
            print(
                "early stopping triggered: "
                f"best_valid_loss={best_valid_loss:.4f}"
            )
            break

In [ ]:
history

## Check Overfitting

In [ ]:
import matplotlib.pyplot as plt

epochs = [item["epoch"] for item in history]
train_losses = [item["train_loss"] for item in history]
valid_losses = [item["valid_loss"] for item in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

axes[0].plot(epochs, train_losses, marker="o", color="tab:blue")
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, valid_losses, marker="o", color="tab:orange")
axes[1].set_title("Valid Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()